<a href="https://colab.research.google.com/github/Ayush-Singh-36/Summer_Training_project/blob/main/electric_vehicle_price_model.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Model Development

# Importing all the required libraries and dependencies

In [15]:
%pip install scikit-learn==1.9.0
import sklearn
print(sklearn.__version__)
from sklearn.preprocessing import OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
import pandas as pd
%pip install category_encoders
import category_encoders as ce

1.9.0


**Looking at data for once**

In [16]:
data = pd.read_csv("Electric_Vehicle_Population_Data.csv")
data.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 138779 entries, 0 to 138778
Data columns (total 17 columns):
 #   Column                                             Non-Null Count   Dtype  
---  ------                                             --------------   -----  
 0   VIN (1-10)                                         138779 non-null  object 
 1   County                                             138776 non-null  object 
 2   City                                               138776 non-null  object 
 3   State                                              138779 non-null  object 
 4   Postal Code                                        138776 non-null  float64
 5   Model Year                                         138779 non-null  int64  
 6   Make                                               138779 non-null  object 
 7   Model                                              138493 non-null  object 
 8   Electric Vehicle Type                              138779 non-null  object

# Dropping unwanted columns

In [17]:
import unittest
import sys
import os
import pandas as pd
import numpy as np
import pickle as pk
data = data.drop(columns = ["VIN (1-10)", "Postal Code", "DOL Vehicle ID", "Vehicle Location", "Electric Utility", "2020 Census Tract", "Legislative District"])
data.info()

class TestPipelineStep1_DataDrop(unittest.TestCase):
    def test_dropped_columns(self):
        test_data = data
        dropped_cols = ["VIN (1-10)", "Postal Code", "DOL Vehicle ID", "Vehicle Location", "Electric Utility", "2020 Census Tract", "Legislative District"]
        for col in dropped_cols:
            self.assertNotIn(col, test_data.columns)
        self.assertIn("Base MSRP", test_data.columns)

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 138779 entries, 0 to 138778
Data columns (total 10 columns):
 #   Column                                             Non-Null Count   Dtype 
---  ------                                             --------------   ----- 
 0   County                                             138776 non-null  object
 1   City                                               138776 non-null  object
 2   State                                              138779 non-null  object
 3   Model Year                                         138779 non-null  int64 
 4   Make                                               138779 non-null  object
 5   Model                                              138493 non-null  object
 6   Electric Vehicle Type                              138779 non-null  object
 7   Clean Alternative Fuel Vehicle (CAFV) Eligibility  138779 non-null  object
 8   Electric Range                                     138779 non-null  int64 
 9   Base

# Dividing the training columns with target columns

In [18]:
x = data[["County", "City", "State", "Model Year", "Make", "Model", "Electric Vehicle Type", "Clean Alternative Fuel Vehicle (CAFV) Eligibility", "Electric Range"]].copy()
y = data["Base MSRP"]

class TestPipelineStep2_SplittingXY(unittest.TestCase):
    def test_split_x_y(self):
        test_x = x
        test_y = y
        self.assertEqual(test_x.shape[1], 9)
        self.assertEqual(len(test_y), len(test_x))

# Checking at every different options for all individual columns

In [19]:
def profile_dataset_features(df: pd.DataFrame, max_categories_to_print: int = 10):
    print(f"=== Dataset Shape: {df.shape[0]} rows | {df.shape[1]} columns ===\n")

    categorical_cols = df.select_dtypes(include=['object', 'category', 'bool']).columns
    numerical_cols = df.select_dtypes(include=[np.number]).columns

    print(f"Found {len(categorical_cols)} Categorical columns and {len(numerical_cols)} Numerical columns.\n")
    print("-" * 50)
    print("CATEGORICAL FEATURE OPTIONS DISCOVERY")
    print("-" * 50)

    for col in categorical_cols:
        unique_vals = df[col].dropna().unique()
        num_unique = len(unique_vals)
        missing_count = df[col].isna().sum()

        print(f"\n🔹 Feature: '{col}' | Unique Values Count: {num_unique} | Missing: {missing_count} rows")

        if num_unique > 30:
            print(f"  ⚠️ High Cardinality! Showing first 5 options sample: {list(unique_vals[:5])}...")
        else:
            value_counts = df[col].value_counts(dropna=False)
            for val, count in value_counts.items():
                pct = (count / len(df)) * 100
                print(f"  - [{val}]: {count} occurrences ({pct:.2f}%)")

    print("\n" + "-" * 50)
    print("NUMERICAL FEATURE BOUNDARY DISCOVERY")
    print("-" * 50)

    numeric_summary = df[numerical_cols].describe().T[['min', 'max', 'mean']]
    print(numeric_summary.to_string())

df = pd.read_csv("Electric_Vehicle_Population_Data.csv")
profile_dataset_features(df)

=== Dataset Shape: 138779 rows | 17 columns ===

Found 10 Categorical columns and 7 Numerical columns.

--------------------------------------------------
CATEGORICAL FEATURE OPTIONS DISCOVERY
--------------------------------------------------

🔹 Feature: 'VIN (1-10)' | Unique Values Count: 9211 | Missing: 0 rows
  ⚠️ High Cardinality! Showing first 5 options sample: ['1N4AZ0CP5D', '1N4AZ1CP8K', '5YJXCAE28L', 'SADHC2S1XK', 'JN1AZ0CP9B']...

🔹 Feature: 'County' | Unique Values Count: 173 | Missing: 3 rows
  ⚠️ High Cardinality! Showing first 5 options sample: ['Kitsap', 'King', 'Thurston', 'Snohomish', 'Yakima']...

🔹 Feature: 'City' | Unique Values Count: 655 | Missing: 3 rows
  ⚠️ High Cardinality! Showing first 5 options sample: ['Bremerton', 'Port Orchard', 'Seattle', 'Olympia', 'Everett']...

🔹 Feature: 'State' | Unique Values Count: 45 | Missing: 0 rows
  ⚠️ High Cardinality! Showing first 5 options sample: ['WA', 'NY', 'CA', 'AP', 'MD']...

🔹 Feature: 'Make' | Unique Values Count

## Handling Missing Values/Dealing with high Cardinality in Features (`x`)

In [20]:
print(x.isnull().sum())

x['County'] = x['County'].fillna(x['County'].mode()[0])
x['City'] = x['City'].fillna(x['City'].mode()[0])
x['Model'] = x['Model'].fillna(x['Model'].mode()[0])

print('\nMissing values after imputation:')
print(x.isnull().sum())

class TestPipelineStep3_Imputation(unittest.TestCase):
    def test_imputations(self):
        test_x = data[["County", "City", "State", "Model Year", "Make", "Model", "Electric Vehicle Type", "Clean Alternative Fuel Vehicle (CAFV) Eligibility", "Electric Range"]].copy()
        test_x['County'] = test_x['County'].fillna(test_x['County'].mode()[0])
        test_x['City'] = test_x['City'].fillna(test_x['City'].mode()[0])
        test_x['Model'] = test_x['Model'].fillna(test_x['Model'].mode()[0])
        self.assertEqual(test_x['County'].isnull().sum(), 0)
        self.assertEqual(test_x['City'].isnull().sum(), 0)
        self.assertEqual(test_x['Model'].isnull().sum(), 0)

County                                                 3
City                                                   3
State                                                  0
Model Year                                             0
Make                                                   0
Model                                                286
Electric Vehicle Type                                  0
Clean Alternative Fuel Vehicle (CAFV) Eligibility      0
Electric Range                                         0
dtype: int64

Missing values after imputation:
County                                               0
City                                                 0
State                                                0
Model Year                                           0
Make                                                 0
Model                                                0
Electric Vehicle Type                                0
Clean Alternative Fuel Vehicle (CAFV) Eligibility    0


## Encoding Categorical Features (Creating `x_processed`)



In [21]:
non_null_indices = y.dropna().index
x = x.loc[non_null_indices]
y = y.loc[non_null_indices]

low_cardinality_cols = ['Electric Vehicle Type', 'Clean Alternative Fuel Vehicle (CAFV) Eligibility']
high_cardinality_cols = ['County', 'City', 'State', 'Make', 'Model']

preprocessor = ColumnTransformer(
    transformers=[
        ('onehot', OneHotEncoder(handle_unknown='ignore'), low_cardinality_cols),
        ('target', ce.TargetEncoder(), high_cardinality_cols)
    ],
    remainder='passthrough'
)

from scipy.sparse import csr_matrix

x_encoded = csr_matrix(preprocessor.fit_transform(x, y)).toarray()

onehot_feature_names = preprocessor.named_transformers_['onehot'].get_feature_names_out(low_cardinality_cols)
target_feature_names = high_cardinality_cols
original_numerical_cols = [col for col in x.columns if col not in low_cardinality_cols + high_cardinality_cols]

all_feature_names = list(onehot_feature_names) + list(target_feature_names) + original_numerical_cols

x_processed = pd.DataFrame(x_encoded, columns=all_feature_names, index=x.index)
display(x_processed.head())

class TestPipelineStep4_EncodingAndScaling(unittest.TestCase):
    def test_processed_shapes(self):
        self.assertIsInstance(x_processed, pd.DataFrame)
        self.assertFalse(x_processed.isnull().any().any())
        self.assertEqual(x_processed.shape[0], len(non_null_indices))

,Electric Vehicle Type_Battery Electric Vehicle (BEV),Electric Vehicle Type_Plug-in Hybrid Electric Vehicle (PHEV),Clean Alternative Fuel Vehicle (CAFV) Eligibility_Clean Alternative Fuel Vehicle Eligible,Clean Alternative Fuel Vehicle (CAFV) Eligibility_Eligibility unknown as battery range has not been researched,Clean Alternative Fuel Vehicle (CAFV) Eligibility_Not eligible due to low battery range,County,City,State,Make,Model,Model Year,Electric Range
0,1.0,0.0,1.0,0.0,0.0,1722.980140,1385.238854,1404.488423,0.000000,0.000000,2013.0,75.0
1,1.0,0.0,1.0,0.0,0.0,1722.980140,1527.471698,1404.488423,0.000000,0.000000,2019.0,150.0
2,1.0,0.0,1.0,0.0,0.0,1438.021310,1356.533358,1404.488423,1777.542849,0.000000,2020.0,293.0
3,1.0,0.0,1.0,0.0,0.0,1059.985927,1078.297776,1404.488423,0.000002,0.000002,2019.0,234.0
4,1.0,0.0,1.0,0.0,0.0,1249.687082,1272.202477,1404.488423,0.000000,0.000000,2011.0,73.0


## Scaling Features

In [22]:

scaler = StandardScaler()

x_processed_scaled = scaler.fit_transform(x_processed)

x_processed = pd.DataFrame(x_processed_scaled, columns=x_processed.columns, index=x_processed.index)

print('x_processed after scaling:')
display(x_processed.head())

x_processed after scaling:


,Electric Vehicle Type_Battery Electric Vehicle (BEV),Electric Vehicle Type_Plug-in Hybrid Electric Vehicle (PHEV),Clean Alternative Fuel Vehicle (CAFV) Eligibility_Clean Alternative Fuel Vehicle Eligible,Clean Alternative Fuel Vehicle (CAFV) Eligibility_Eligibility unknown as battery range has not been researched,Clean Alternative Fuel Vehicle (CAFV) Eligibility_Not eligible due to low battery range,County,City,State,Make,Model,Model Year,Electric Range
0,0.54729,-0.54729,1.122754,-0.874788,-0.376452,1.711713,-0.036493,-0.011933,-0.921486,-0.26646,-2.247633,0.027556
1,0.54729,-0.54729,1.122754,-0.874788,-0.376452,1.711713,0.216306,-0.011933,-0.921486,-0.26646,-0.256256,0.794644
2,0.54729,-0.54729,1.122754,-0.874788,-0.376452,0.178162,-0.087513,-0.011933,0.250041,-0.26646,0.075640,2.257225
3,0.54729,-0.54729,1.122754,-0.874788,-0.376452,-1.856294,-0.582037,-0.011933,-0.921486,-0.26646,-0.256256,1.653782
4,0.54729,-0.54729,1.122754,-0.874788,-0.376452,-0.835388,-0.237399,-0.011933,-0.921486,-0.26646,-2.911425,0.007101


## Splitting Data into Training and Testing Sets



In [23]:

X_train, X_test, y_train, y_test = train_test_split(x_processed, y, test_size=0.2, random_state=42)

print(f"X_train shape: {X_train.shape}")
print(f"X_test shape: {X_test.shape}")
print(f"y_train shape: {y_train.shape}")
print(f"y_test shape: {y_test.shape}")

class TestPipelineStep5_TrainTestSplit(unittest.TestCase):
    def test_split_dimensions(self):
        self.assertEqual(X_train.shape[0] + X_test.shape[0], x_processed.shape[0])
        self.assertEqual(X_train.shape[1], X_test.shape[1])

X_train shape: (111023, 12)
X_test shape: (27756, 12)
y_train shape: (111023,)
y_test shape: (27756,)


## Random Forest Regression

In [24]:
from sklearn.ensemble import RandomForestRegressor
import numpy as np
from sklearn.metrics import mean_squared_error, r2_score

regressor_model = RandomForestRegressor(n_estimators=100, random_state=42)
regressor_model.fit(X_train, y_train)

y_pred_prices = regressor_model.predict(X_test)
print(f"First 10 predicted prices: {y_pred_prices[:10]}")

mse = mean_squared_error(y_test, y_pred_prices)
r2 = r2_score(y_test, y_pred_prices)

print(f"Test Mean Squared Error: {mse:.2f}")
print(f"Test R2 Score: {r2:.4f}")

class TestPipelineStep6_Regression(unittest.TestCase):
    def test_regressor_predictions(self):
        self.assertEqual(len(y_pred_prices), len(X_test))
        self.assertTrue(np.all(y_pred_prices >= 0))
        self.assertGreater(r2, -0.5)

test_case = TestPipelineStep6_Regression()
test_case.test_regressor_predictions()
print("✅ Regression model unit test passed successfully!")

First 10 predicted prices: [0. 0. 0. 0. 0. 0. 0. 0. 0. 0.]
Test Mean Squared Error: 3104618.19
Test R2 Score: 0.9636
✅ Regression model unit test passed successfully!


# Making prediction over trained model

In [25]:
y_pred_prices = regressor_model.predict(X_test)
print(f"First 10 predicted prices: {y_pred_prices[:10]}")

class TestPipelineStep7_Predictions(unittest.TestCase):
    def test_predictions_shape(self):
        self.assertEqual(len(y_pred_prices), len(X_test))
        self.assertTrue(np.all(y_pred_prices >= 0))

test_case = TestPipelineStep7_Predictions()
test_case.test_predictions_shape()
print("✅ Prediction step unit test passed successfully!")

First 10 predicted prices: [0. 0. 0. 0. 0. 0. 0. 0. 0. 0.]
✅ Prediction step unit test passed successfully!


# Evaluating Model's Performance

In [26]:
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

mae_train = mean_absolute_error(y_train, regressor_model.predict(X_train))
r2_train = r2_score(y_train, regressor_model.predict(X_train))
print(f"Training R2 Score: {r2_train:.4f}")

print("\n--- Evaluating Test Set Performance ---")
mse_test = mean_squared_error(y_test, y_pred_prices)
r2_test = r2_score(y_test, y_pred_prices)
print(f"Test Mean Squared Error: {mse_test:.2f}")
print(f"Test R2 Score: {r2_test:.4f}")

class TestPipelineStep8_EvaluationMetrics(unittest.TestCase):
    def test_metrics_validity(self):
        self.assertGreaterEqual(r2_test, -1.0)
        self.assertLessEqual(r2_test, 1.0)
        self.assertTrue(mse_test >= 0.0)

test_case = TestPipelineStep8_EvaluationMetrics()
test_case.test_metrics_validity()
print("✅ Evaluation metrics unit test passed successfully!")

Training R2 Score: 0.9876

--- Evaluating Test Set Performance ---
Test Mean Squared Error: 3104618.19
Test R2 Score: 0.9636
✅ Evaluation metrics unit test passed successfully!


# Saving Model Artifacts with help of pickle

In [27]:
import pickle as pk

model_artifacts = {
    'preprocessor': preprocessor,
    'scaler': scaler,
    'model': regressor_model
}

artifact_filename = 'model_artifact.pkl'
with open(artifact_filename, 'wb') as f:
    pk.dump(model_artifacts, f)

print(f"All model artifacts saved to '{artifact_filename}'")

class TestPipelineStep9_SavedArtifacts(unittest.TestCase):
    def test_pickle_file_exists_and_loads(self):
        self.assertTrue(os.path.exists('model_artifact.pkl'))
        with open('model_artifact.pkl', 'rb') as f:
            artifacts = pk.load(f)
        self.assertIn('preprocessor', artifacts)
        self.assertIn('scaler', artifacts)
        self.assertIn('model', artifacts)

test_case = TestPipelineStep9_SavedArtifacts()
test_case.test_pickle_file_exists_and_loads()
print("✅ Saved artifacts unit test passed successfully!")

All model artifacts saved to 'model_artifact.pkl'
✅ Saved artifacts unit test passed successfully!


# Testing all the cells

In [28]:
suite = unittest.TestSuite()
loader = unittest.TestLoader()

test_classes = [
    TestPipelineStep1_DataDrop,
    TestPipelineStep2_SplittingXY,
    TestPipelineStep3_Imputation,
    TestPipelineStep4_EncodingAndScaling,
    TestPipelineStep5_TrainTestSplit,
    TestPipelineStep6_Regression,
    TestPipelineStep7_Predictions,
    TestPipelineStep8_EvaluationMetrics,
    TestPipelineStep9_SavedArtifacts
]

for test_class in test_classes:
    suite.addTest(loader.loadTestsFromTestCase(test_class))

runner = unittest.TextTestRunner(verbosity=2)
runner.run(suite)

test_dropped_columns (__main__.TestPipelineStep1_DataDrop.test_dropped_columns) ... ok
test_split_x_y (__main__.TestPipelineStep2_SplittingXY.test_split_x_y) ... ok
test_imputations (__main__.TestPipelineStep3_Imputation.test_imputations) ... ok
test_processed_shapes (__main__.TestPipelineStep4_EncodingAndScaling.test_processed_shapes) ... ok
test_split_dimensions (__main__.TestPipelineStep5_TrainTestSplit.test_split_dimensions) ... ok
test_regressor_predictions (__main__.TestPipelineStep6_Regression.test_regressor_predictions) ... ok
test_predictions_shape (__main__.TestPipelineStep7_Predictions.test_predictions_shape) ... ok
test_metrics_validity (__main__.TestPipelineStep8_EvaluationMetrics.test_metrics_validity) ... ok
test_pickle_file_exists_and_loads (__main__.TestPipelineStep9_SavedArtifacts.test_pickle_file_exists_and_loads) ... ok

----------------------------------------------------------------------
Ran 9 tests in 0.125s

OK


<unittest.runner.TextTestResult run=9 errors=0 failures=0>